In [5]:
import numpy as np
import cv2 as cv
import os
import glob
import sys
sys.path.append("../")
from PIL import Image
from matplotlib import pyplot as plt
from map.quadtree import FixedQuadTree
from map.transform import ImagePatchify

In [6]:
def get_image(root_dir):
    classes = sorted(os.listdir(root_dir))  # Get class directories
    image_paths = []
    for cls_name in classes:
        cls_dir = os.path.join(root_dir, cls_name)
        for img_path in glob.glob(os.path.join(cls_dir, "*.JPEG")):  # Adjust extension if needed
            image_paths.append(img_path)
    return image_paths

def patchify(img, fixed_length=196, patch_size=16, smooth_factor=5, canny=[50, 100]):
    grey_img = cv.GaussianBlur(img, (smooth_factor, smooth_factor), 0)
    edges = cv.Canny(grey_img, canny[0], canny[1])
    qdt = FixedQuadTree(domain=edges, fixed_length=fixed_length)
    seq_img, seq_size, seq_pos = qdt.serialize(img, size=(patch_size,patch_size,3))
    seq_size = np.asarray(seq_size)
    seq_img = np.asarray(seq_img)
    seq_img = np.reshape(seq_img, [patch_size*patch_size, -1, 3])
    return seq_img, seq_size, seq_pos, qdt, grey_img, edges

In [10]:
data_path = '/Users/zhangenzhi/Desktop/BraTS-MET-00001-000-t1c.npy'
img_data = np.load(data_path)
mean = img_data.mean()
print("MEAN", mean)
std = img_data.std()
print("STD", std)
upper_bound = np.quantile(img_data,.995)
print("UPPER_BOUND", upper_bound)
lower_bound = np.quantile(img_data,.005)
print("LOWER_BOUND", lower_bound)
img_data = np.clip(img_data, lower_bound, upper_bound)
img_data = (img_data - mean) / (std + 1e-8)
img_data = (img_data - np.min(img_data)) / ((np.max(img_data) - np.min(img_data)) + 1e-8)

# img_data1 = np.load(data_path+"/BraTS-MET-00001-000-t1c.npy")
# img_data2 = np.load(data_path+"/BraTS-MET-00001-000-t1n.npy")
# img_data3 = np.load(data_path+"/BraTS-MET-00001-000-t2w.npy")
# img_data4 = np.load(data_path+"/BraTS-MET-00001-000-t2f.npy")

img_size_x = 64
img_size_y = 64
img_size_z = 64
#x_start = 70
x_start = 0
#y_start = 80
y_start = 0
#z_start = 60
z_start = 0

print("DATA_SHAPE",img_data.shape)
breaker1 = False
breaker2 = False
for i in range(img_data.shape[0]-64):
    x_start = i
    for j in range(img_data.shape[1]-64):
        y_start = j
    
        for k in range(img_data.shape[2]-1):
            z_start = k
            #print(x_start)
            #img_slice = img_data[x_start:x_start+img_size_x, y_start:y_start+img_size_y, z_start:z_start+img_size_z]
            img_slice = img_data[x_start:x_start+img_size_x, y_start:y_start+img_size_y, z_start:z_start+1]
            #if np.count_nonzero(img_slice) > (1.0/3072.0)*(64*64):
            #if np.count_nonzero(img_slice) > 2:
            if np.count_nonzero(img_slice) > 1024 and np.count_nonzero(img_slice) < 2048:
                print("X_START, Y_START, Z_start", x_start, y_start, z_start)
                breaker1 = True
                breaker2 = True
                break
        if breaker1:
            break
    if breaker2:
        break
print("SHAPE_IMAGE_SLICE",img_slice.shape)

# img_slice1 = img_data1[x_start:x_start+img_size_x, y_start:y_start+img_size_y, z_start:z_start+img_size_z]
# img_slice2 = img_data2[x_start:x_start+img_size_x, y_start:y_start+img_size_y, z_start:z_start+img_size_z]
# img_slice3 = img_data3[x_start:x_start+img_size_x, y_start:y_start+img_size_y, z_start:z_start+img_size_z]
# img_slice4 = img_data4[x_start:x_start+img_size_x, y_start:y_start+img_size_y, z_start:z_start+img_size_z]

# img_slice1 = img_slice1[:,:,0]
# img_slice2 = img_slice2[:,:,0]
# img_slice3 = img_slice4[:,:,0]
# img_slice4 = img_slice4[:,:,0]

# img_slice_stack = np.stack([img_slice1,img_slice2,img_slice3,img_slice4])

smooth_factor = 3
#smooth_factor = 0

print("MIN", np.min(img_slice))
print("MAX", np.max(img_slice))
print("NNZ", np.count_nonzero(img_slice))

MEAN 23.197039
STD 57.49831
UPPER_BOUND 194.9759
LOWER_BOUND 0.0
DATA_SHAPE (240, 240, 155)
X_START, Y_START, Z_start 6 99 61
SHAPE_IMAGE_SLICE (64, 64, 1)
MIN 0.0
MAX 1.0
NNZ 1025


In [15]:
img_slice.shape

(64, 64, 1)

In [13]:
image = img_slice
seq_img, seq_size, seq_pos, qdt, grey_img, edges= patchify(image, fixed_length=64, patch_size=4,smooth_factor=smooth_factor)

error: OpenCV(4.11.0) /Users/xperience/GHA-Actions-OpenCV/_work/opencv-python/opencv-python/opencv/modules/imgproc/src/canny.cpp:829: error: (-215:Assertion failed) _src.depth() == CV_8U in function 'Canny'


In [4]:
sum(seq_size)

9980

In [5]:
seq_size

array([8, 8, 8, ..., 4, 8, 8])